# 2018 年违规编码核查（独立补充）

本文件保留正文精简时移出的核查，可从项目根目录独立运行。它不参与预测特征构造。

In [ ]:
from pathlib import Path
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/food_inspections_2016-01-01_2026-10-30.csv').exists())
df = pd.read_csv(root / 'data/food_inspections_2016-01-01_2026-10-30.csv', dtype='string')
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'], format='%m/%d/%Y')

In [51]:
reform_audit = pd.DataFrame({'period': df['Inspection Date'].ge('2018-07-01')})
reform_audit['period'] = reform_audit['period'].map({False: 'Before July 2018', True: 'From July 2018'})
reform_audit['has_text'] = df['Violations'].notna()
reform_audit['has_code_46_63'] = df['Violations'].str.contains(
    r'(?:^|\|)\s*(?:4[6-9]|5\d|6[0-3])\.', regex=True, na=False)
reform_audit.groupby('period').agg(all_records=('has_text', 'size'),
    with_violation_text=('has_text', 'sum'), with_code_46_63=('has_code_46_63', 'sum'))

,all_records,with_violation_text,with_code_46_63
period,,,
Before July 2018,53800,40380,0
From July 2018,144082,101093,91750


改制前 **53,800** 条记录中，没有匹配到编号 46–63；改制后 **144,082** 条记录中，有 **91,750** 条匹配到。这支持报告编码发生变化，但不能单独衡量监管松紧或食品安全改善；两个阶段时长不同，也不能直接比较违规数量。改制前数据是否仍有预测价值，应通过相同未来验证窗口下的实验判断。